In [29]:
import numpy as np
from scipy.stats import multivariate_normal
import torch
import pandas as pd

In [30]:
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [31]:
def generate_mesh_and_z(sigma1, sigma2, sigma3):
    """
    Generates a mesh grid and corresponding probability density values for a mixture
    of three 2D Gaussian distributions.

    Args:
    sigma1 (torch.Tensor): The covariance tensor for the first Gaussian distribution.
    sigma2 (torch.Tensor): The covariance tensor for the second Gaussian distribution.
    sigma3 (torch.Tensor): The covariance tensor for the third Gaussian distribution.

    Returns:
    tuple: Tuple containing mesh grid arrays (xx, yy) and the corresponding probability
    density values (zz).
    """
    N_points = 10
    xl = -2
    xr = 2

    x = np.linspace(xl, xr, N_points+1)
    xx, yy = np.meshgrid(x, x)

    X = torch.tensor(np.column_stack([xx.ravel(), yy.ravel()]), dtype=torch.float32)

    mu1 = torch.tensor([1, 1], dtype=torch.float32)
    mu2 = torch.tensor([1, -1], dtype=torch.float32)
    mu3 = torch.tensor([-1, -1], dtype=torch.float32)

    z1 = torch.tensor(multivariate_normal.pdf(X.numpy(), mu1.numpy(), sigma1.numpy()), dtype=torch.float32)
    z2 = torch.tensor(multivariate_normal.pdf(X.numpy(), mu2.numpy(), sigma2.numpy()), dtype=torch.float32)
    z3 = torch.tensor(multivariate_normal.pdf(X.numpy(), mu3.numpy(), sigma3.numpy()), dtype=torch.float32)

    zz = (z1 + z2 + z3).reshape(xx.shape)

    return xx, yy, zz

In [32]:
sigma1_d = 0.15 * torch.eye(2)
sigma2_d = 0.2 * torch.eye(2)
sigma3_d = 0.3 *torch.eye(2)

In [33]:
xx, yy, zz = generate_mesh_and_z(sigma1_d, sigma2_d, sigma3_d)

In [34]:
fig = go.Figure(data=[go.Surface(z=zz.numpy(), x=xx, y=yy)])
fig.update_layout(scene=dict(aspectmode='data'))
fig.update_layout(scene=dict(camera=dict(eye=dict(x=2, y=2, z=1))))

#fig.show()

In [35]:
X = torch.tensor(np.column_stack([xx.ravel(), yy.ravel()]), dtype=torch.float32) # Total of samples in 2D Gaussian distributions

In [36]:
y = torch.tensor(np.column_stack([zz.ravel()]), dtype=torch.float32) # corresponding probability density values

In [37]:
T = 4 # Total of segments per dimension. T^2 is the amount of sub-blocks in the partition

#Escaled and normalized samples
x_norm = ( X - X.min() ) / ( ( X.max() - X.min() ) / T )

In [38]:
#Get Squeeze Factor for Output values
def squeze_factor(Y):
  e_f   = 0.0
  max_y = Y.max()
  if max_y > 1:
    e_f = 1/max_y
  else:
    e_f = 1.0
  return e_f

In [40]:
y = y*squeze_factor(y) # scaled Y

In [41]:
t  = x_norm.numpy().astype(int) # The integer part of the normalized samples

In [42]:
x_n = x_norm - t # Normalized samples from 0 to 1

In [43]:

class SubBlock:
    """
    Represents a sub-block in a 2D grid.

    Attributes:
    - vertices (np.ndarray): The vertices of the sub-block.
    - samples_inside (list): List of samples inside the sub-block.
    - output_values (list): List of output values.

    Methods:
    - add_point(point): Add a point to the sub-block.
    """
    def __init__(self, vertices):
        self.vertices = vertices
        self.samples_inside  = []
        self.output_values = []

    def add_point(self, point, y):
        self.samples_inside.append(point)
        self.output_values.append(y)

def get_sub_block_vertices(grid_size, row, col):
    """
    Get the vertices of a sub-block in a 2D grid.

    Args:
    - grid_size (int): The number of segments per dimension.
    - row (int): The row index of the sub-block.
    - col (int): The column index of the sub-block.

    Returns:
    np.ndarray: The vertices of the sub-block.
    """
    delta = 1.0 / grid_size
    x0 = col * delta
    x1 = (col + 1) * delta
    y0 = row * delta
    y1 = (row + 1) * delta
    return np.array([[x0, y0], [x1, y0], [x0, y1], [x1, y1]])


def locate_samples_in_sub_blocks(x_n, y, t, grid_size):
    """
    Locate points in their respective sub-blocks in a 2D grid.

    Args:
    - x_n (np.ndarray): The normalized points between 0 and 1.
    - y (np.narray) : The output values associated with the samples
    - t (np.ndarray): The integer part of the normalized points.
    - grid_size (int): The number of segments per dimension.

    Returns:
    np.ndarray: Array of SubBlock instances representing the sub-blocks.
    """

    sub_blocks = np.empty((grid_size, grid_size), dtype=object)

    for row in range(grid_size):
        for col in range(grid_size):
            sub_blocks[row, col] = SubBlock(get_sub_block_vertices(grid_size, row, col))

    for i in range(x_n.shape[0]):
        point = x_n[i]
        location = t[i]
        print(location)
        if location[0] > grid_size - 1:
          continue
        if location[1] > grid_size - 1:
          continue

        sub_block = sub_blocks[location[0], location[1]]
        sub_block.add_point(point, y[i])

    return sub_blocks



sub_blocks = locate_samples_in_sub_blocks(x_n.numpy(), y.numpy(), t, T)


[0 0]
[0 0]
[0 0]
[1 0]
[1 0]
[2 0]
[2 0]
[2 0]
[3 0]
[3 0]
[4 0]
[0 0]
[0 0]
[0 0]
[1 0]
[1 0]
[2 0]
[2 0]
[2 0]
[3 0]
[3 0]
[4 0]
[0 0]
[0 0]
[0 0]
[1 0]
[1 0]
[2 0]
[2 0]
[2 0]
[3 0]
[3 0]
[4 0]
[0 1]
[0 1]
[0 1]
[1 1]
[1 1]
[2 1]
[2 1]
[2 1]
[3 1]
[3 1]
[4 1]
[0 1]
[0 1]
[0 1]
[1 1]
[1 1]
[2 1]
[2 1]
[2 1]
[3 1]
[3 1]
[4 1]
[0 2]
[0 2]
[0 2]
[1 2]
[1 2]
[2 2]
[2 2]
[2 2]
[3 2]
[3 2]
[4 2]
[0 2]
[0 2]
[0 2]
[1 2]
[1 2]
[2 2]
[2 2]
[2 2]
[3 2]
[3 2]
[4 2]
[0 2]
[0 2]
[0 2]
[1 2]
[1 2]
[2 2]
[2 2]
[2 2]
[3 2]
[3 2]
[4 2]
[0 3]
[0 3]
[0 3]
[1 3]
[1 3]
[2 3]
[2 3]
[2 3]
[3 3]
[3 3]
[4 3]
[0 3]
[0 3]
[0 3]
[1 3]
[1 3]
[2 3]
[2 3]
[2 3]
[3 3]
[3 3]
[4 3]
[0 4]
[0 4]
[0 4]
[1 4]
[1 4]
[2 4]
[2 4]
[2 4]
[3 4]
[3 4]
[4 4]


In [45]:
#check values
for row in sub_blocks:
  for block in row:
    print(f'\n Block vertices: {block.vertices}')
    print(f'Block samples inside: {block.samples_inside} \n')
    print(f'Block output values: {block.output_values} \n')


 Block vertices: [[0.   0.  ]
 [0.25 0.  ]
 [0.   0.25]
 [0.25 0.25]]
Block samples inside: [array([0., 0.]), array([0.39999998, 0.        ]), array([0.79999995, 0.        ]), array([0.        , 0.39999998]), array([0.39999998, 0.39999998]), array([0.79999995, 0.39999998]), array([0.        , 0.79999995]), array([0.39999998, 0.79999995]), array([0.79999995, 0.79999995])] 

Block output values: [array([0.01892564], dtype=float32), array([0.05499182], dtype=float32), array([0.09373968], dtype=float32), array([0.05499182], dtype=float32), array([0.15978849], dtype=float32), array([0.272378], dtype=float32), array([0.09373932], dtype=float32), array([0.27237624], dtype=float32), array([0.4642978], dtype=float32)] 


 Block vertices: [[0.25 0.  ]
 [0.5  0.  ]
 [0.25 0.25]
 [0.5  0.25]]
Block samples inside: [array([0.        , 0.20000005]), array([0.39999998, 0.20000005]), array([0.79999995, 0.20000005]), array([0.        , 0.60000002]), array([0.39999998, 0.60000002]), array([0.79999995, 